# Improved product categorisation (keyword + regex rules)

This notebook reads `products_cl.csv`, builds a normalised text field from **name + desc**, applies an ordered set of keyword/regex rules, and outputs an improved categorised CSV.

**Outputs**
- `products_cl_categorised_improved.csv` with two new columns:
  - `category_1` (broad)
  - `category_2` (specific)

> Tip: Add / adjust patterns in the `rules` list to improve precision for your catalogue.


In [1]:
# Imports
import pandas as pd
import re
import unicodedata


In [2]:
# 1. Get the data from the URL
url = "https://drive.google.com/file/d/1s7Lai4NSlsYjGEPg1QSOUJobNYVsZBOJ/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]

# 2. Load it into a DataFrame
products_cl = pd.read_csv(path)

# 3. Simplify: Just point 'df' to your existing DataFrame
df = products_cl

# View the results
print(df.head())

       sku                                           name  \
0  RAI0007              Silver Rain Design mStand Support   
1  APP0023              Apple Mac Keyboard Keypad Spanish   
2  APP0025               Mighty Mouse Apple Mouse for Mac   
3  APP0072  Apple Dock to USB Cable iPhone and iPod white   
4  KIN0007    Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM   

                                                desc  price  in_stock  \
0       Aluminum support compatible with all MacBook  59.99         1   
1          USB ultrathin keyboard Apple Mac Spanish.  59.00         0   
2                             mouse Apple USB cable.  59.00         0   
3              IPhone dock and USB Cable Apple iPod.  25.00         0   
4  2GB RAM Mac mini and iMac (2006/07) MacBook Pr...  34.99         1   

       type  
0      8696  
1  13855401  
2      1387  
3      1230  
4      1364  


## 1) Normalise text

We combine `name` + `desc` into a single searchable string and normalise it:
- lowercase
- remove accents/diacritics
- strip punctuation (keeping numbers and a few connector chars)
- collapse whitespace


In [3]:
def normalize(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)

    # Remove accents/diacritics (e.g. é -> e)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

    # Lowercase
    text = text.lower()

    # Keep letters, digits, spaces and some connector characters
    text = re.sub(r"[^a-z0-9\s\-\+\/]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_norm"] = (df.get("name", "").fillna("") + " " + df.get("desc", "").fillna("")).map(normalize)
df[["name", "desc", "text_norm"]].head()


,name,desc,text_norm
0,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,silver rain design mstand support aluminum sup...
1,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,apple mac keyboard keypad spanish usb ultrathi...
2,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,mighty mouse apple mouse for mac mouse apple u...
3,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,apple dock to usb cable iphone and ipod white ...
4,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,mac memory kingston 2gb 667mhz ddr2 so-dimm 2g...


## 2) Categorisation rules

Rules are applied **top-to-bottom** (first match wins).  
Each rule has:

- `category_1` (broad)
- `category_2` (specific)
- a list of regex patterns to match in the normalised text

Adjust this list to fit your taxonomy.


In [4]:
rules = [
    # Computers
    ("Computers", "Mac mini", [r"\bmac mini\b"]),
    ("Computers", "iMac", [r"\bimac\b"]),
    ("Computers", "Mac Pro", [r"\bmac pro\b"]),
    ("Computers", "MacBook", [r"\bmacbook\b", r"\bmac book\b"]),

    # Mobile
    ("Mobile Devices", "iPhone", [r"\biphone\b"]),
    ("Mobile Devices", "iPad", [r"\bipad\b"]),
    ("Mobile Devices", "iPod", [r"\bipod\b"]),

    # Wearables
    ("Wearables", "Apple Watch Bands", [
        r"\bapple watch\b.*\b(band|bracelet|strap|correa)\b",
        r"\b(band|bracelet|strap|correa)\b.*\bapple watch\b",
    ]),
    ("Wearables", "Apple Watch Cases", [
        r"\bapple watch\b.*\b(case|housing|shell|protector|cover)\b",
        r"\b(case|housing|shell|protector|cover)\b.*\bapple watch\b",
    ]),
    ("Wearables", "Apple Watch Accessories", [r"\bapple watch\b"]),

    # Audio
    ("Audio", "Headphones & Earphones", [
        r"\bheadphones?\b",
        r"\bearphones?\b",
        r"\bearbuds?\b",
        r"\bairpods\b",
        r"\bin-?ear\b",
        r"\bearpod(s)?\b",
    ]),
    ("Audio", "Speakers", [r"\bspeakers?\b", r"\baltavoz(es)?\b"]),
    ("Audio", "Microphones", [r"\bmicrophones?\b", r"\bmic\b"]),

    # Displays
    ("Displays", "Monitors", [r"\bmonitor\b", r"\bdisplay\b"]),

    # Networking
    ("Networking", "Switches", [r"\bswitch\b", r"\bgigabit switch\b"]),
    ("Networking", "Ethernet & Cables", [
        r"\bethernet\b", r"\brj45\b", r"\bcat ?6\b", r"\bcat ?5\b", r"\blan\b"
    ]),
    ("Networking", "Routers & Access Points", [r"\brouter\b", r"\baccess point\b", r"\bairport\b"]),
    ("Networking", "Wi[\-\s]?fi & Bluetooth Adapters", [r"\bwi-?fi\b", r"\bwireless\b", r"\bbluetooth\b"]),

    # Storage & Memory
    ("Storage & Memory", "NAS", [r"\b(nas|synology|qnap)\b", r"\bnetwork storage\b"]),
    ("Storage & Memory", "RAM", [r"\b(ddr\d|so-dimm|sodimm|ram|memory)\b"]),
    ("Storage & Memory", "Hard Drives", [r"\b(hdd|hard drive|disco duro)\b", r"\bwd red\b"]),
    ("Storage & Memory", "SSD", [r"\bssd\b", r"\bsolid state\b"]),
    ("Storage & Memory", "Flash & Cards", [
        r"\b(usb flash|flash drive|pendrive|thumb drive|sd card|micro ?sd|cf card)\b"
    ]),

    # Drives
    ("Drives", "Optical (DVD/Blu-ray)", [r"\bdvd\b", r"\bblu-?ray\b", r"\boptical\b", r"\bcd\b.*\bdrive\b"]),

    # Input devices
    ("Input Devices", "Keyboards", [r"\bkeyboard\b", r"\bteclado\b"]),
    ("Input Devices", "Mice & Trackpads", [r"\bmouse\b", r"\btrackpad\b", r"\bmagic mouse\b"]),
    ("Input Devices", "Graphics Tablets", [r"\bwacom\b", r"\bintuos\b", r"\bgraphic tablet\b", r"\bpen tablet\b"]),
    ("Input Devices", "Stylus", [r"\b(stylus|pointer|pencil|jot)\b"]),

    # Cables & Adapters
    ("Cables & Adapters", "Lightning & Dock", [r"\blightning\b", r"\bdock\b"]),
    ("Cables & Adapters", "USB", [r"\busb\b"]),
    ("Cables & Adapters", "HDMI/Video", [
        r"\bhdmi\b", r"\bdisplayport\b", r"\bdvi\b", r"\bvga\b",
        r"\bthunderbolt\b", r"\bmini displayport\b"
    ]),

    # Power & Charging
    ("Power & Charging", "Chargers & Adapters", [r"\bcharger\b", r"\bpower adapter\b", r"\bac adapter\b", r"\bcharg(e|ing)\b"]),
    ("Power & Charging", "Batteries & Power Banks", [r"\bbatter(y|ies)\b", r"\bpower bank\b", r"\bexternal battery\b"]),

    # Cases & protection
    ("Cases & Protection", "Phone Cases", [r"\bcase\b.*\biphone\b", r"\biphone\b.*\bcase\b", r"\bcover\b.*\biphone\b"]),
    ("Cases & Protection", "Tablet Cases", [r"\bcase\b.*\bipad\b", r"\bipad\b.*\bcase\b", r"\bfolio\b", r"\bsleeve\b"]),
    ("Cases & Protection", "Screen Protectors", [r"\bscreen protector\b", r"\btempered glass\b", r"\bglass\b.*\bprotector\b"]),

    # Mounts & stands
    ("Mounts & Stands", "Stands & Holders", [r"\bstand\b", r"\bmstand\b", r"\bholder\b", r"\bmount\b", r"\barmband\b", r"\bactionsleeve\b"]),
]

compiled_rules = [(c1, c2, [re.compile(p) for p in patterns]) for c1, c2, patterns in rules]
len(compiled_rules)


<>:45: SyntaxWarning: invalid escape sequence '\-'
<>:45: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_2296/3316927342.py:45: SyntaxWarning: invalid escape sequence '\-'
  ("Networking", "Wi[\-\s]?fi & Bluetooth Adapters", [r"\bwi-?fi\b", r"\bwireless\b", r"\bbluetooth\b"]),


37

## 3) Apply rules (first match wins)

If nothing matches, we label it as `Other / Other`.


In [5]:
def categorize(text_norm: str):
    for c1, c2, patterns in compiled_rules:
        for pat in patterns:
            if pat.search(text_norm):
                return c1, c2
    return "Other", "Other"

cats = df["text_norm"].map(categorize)
df["category_1"] = [c[0] for c in cats]
df["category_2"] = [c[1] for c in cats]

df[["name", "desc", "category_1", "category_2"]].head()


,name,desc,category_1,category_2
0,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,Computers,MacBook
1,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,Input Devices,Keyboards
2,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,Input Devices,Mice & Trackpads
3,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,Mobile Devices,iPhone
4,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,Computers,Mac mini


## 4) Export + sanity checks

In [6]:
# 1. Configuration
OUTPUT_PATH = "products_cl_categorised_improved.csv"

# 2. Cleanup
# We use errors='ignore' so the code doesn't crash if you run it twice
out = df.drop(columns=["text_norm"], errors='ignore')

# 3. Save to CSV
out.to_csv(OUTPUT_PATH, index=False)

# 4. Generate Summary
summary = (out.groupby(["category_1", "category_2"])
             .size()
             .sort_values(ascending=False)
             .reset_index(name="count"))

# 5. Final Output (Displaying the top 20 and the path name as requested)
print(f"Success! File saved as: {OUTPUT_PATH}")
summary.head(20), OUTPUT_PATH

Success! File saved as: products_cl_categorised_improved.csv


(           category_1                        category_2  count
 0      Mobile Devices                            iPhone   2917
 1           Computers                           MacBook   1285
 2    Storage & Memory                               NAS   1022
 3      Mobile Devices                              iPad    788
 4           Computers                              iMac    738
 5    Storage & Memory                       Hard Drives    485
 6            Displays                          Monitors    271
 7               Other                             Other    264
 8           Wearables                 Apple Watch Bands    229
 9           Computers                          Mac mini    226
 10          Computers                           Mac Pro    226
 11  Cables & Adapters                               USB    201
 12  Cables & Adapters                        HDMI/Video    180
 13         Networking  Wi[\-\s]?fi & Bluetooth Adapters    161
 14         Networking                 E